### Loading libraries

In [13]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium

from tqdm import tqdm
from pathlib import Path

### Creating Dummy Data

In [15]:
trip_demo = pd.DataFrame({
    "ride_id": list(range(1, 21)),

    "start_station": [
        "Station A", "Station A", "Station A", "Station A", "Station A",
        "Station B", "Station B", "Station B", "Station B",
        "Station C", "Station C", "Station C",
        "Station D", "Station D", "Station D",
        "Station E", "Station E",
        "Station A", "Station B", "Station C"
    ],

    "end_station": [
        "Station B", "Station B", "Station B", "Station C", "Station C",
        "Station A", "Station A", "Station C", "Station D",
        "Station A", "Station B", "Station E",
        "Station A", "Station C", "Station E",
        "Station A", "Station D",
        "Station E", "Station E", "Station D"
    ],

    "started_at": pd.to_datetime([
        "2025-01-01 08:00", "2025-01-01 08:15", "2025-01-01 08:30",
        "2025-01-01 09:00", "2025-01-01 09:20",
        "2025-01-01 10:00", "2025-01-01 10:15", "2025-01-01 10:40",
        "2025-01-01 11:00",
        "2025-01-01 11:30", "2025-01-01 12:00", "2025-01-01 12:20",
        "2025-01-01 13:00", "2025-01-01 13:30", "2025-01-01 14:00",
        "2025-01-01 14:30", "2025-01-01 15:00",
        "2025-01-01 15:30", "2025-01-01 16:00", "2025-01-01 16:30"
    ]),

    "ended_at": pd.to_datetime([
        "2025-01-01 08:10", "2025-01-01 08:28", "2025-01-01 08:42",
        "2025-01-01 09:18", "2025-01-01 09:35",
        "2025-01-01 10:12", "2025-01-01 10:30", "2025-01-01 10:55",
        "2025-01-01 11:18",
        "2025-01-01 11:48", "2025-01-01 12:14", "2025-01-01 12:45",
        "2025-01-01 13:20", "2025-01-01 13:48", "2025-01-01 14:20",
        "2025-01-01 14:55", "2025-01-01 15:22",
        "2025-01-01 15:58", "2025-01-01 16:25", "2025-01-01 16:50"
    ]),

    "member_casual": [
        "member", "member", "casual", "member", "casual",
        "member", "member", "casual", "member",
        "casual", "member", "casual",
        "member", "casual", "member",
        "casual", "member",
        "member", "casual", "member"
    ]
})



In [16]:
trip_demo

,ride_id,start_station,end_station,started_at,ended_at,member_casual
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual
5,6,Station B,Station A,2025-01-01 10:00:00,2025-01-01 10:12:00,member
6,7,Station B,Station A,2025-01-01 10:15:00,2025-01-01 10:30:00,member
7,8,Station B,Station C,2025-01-01 10:40:00,2025-01-01 10:55:00,casual
8,9,Station B,Station D,2025-01-01 11:00:00,2025-01-01 11:18:00,member
9,10,Station C,Station A,2025-01-01 11:30:00,2025-01-01 11:48:00,casual


In [17]:
station_coordinates = pd.DataFrame({
    "station": [
        "Station A",
        "Station B",
        "Station C",
        "Station D",
        "Station E"
    ],
    "lat": [
        40.735,
        40.751,
        40.742,
        40.728,
        40.760
    ],
    "lng": [
        -73.991,
        -73.977,
        -73.985,
        -73.970,
        -73.995
    ]
})



In [25]:
station_coordinates

,station,lat,lng
0,Station A,40.735,-73.991
1,Station B,40.751,-73.977
2,Station C,40.742,-73.985
3,Station D,40.728,-73.970
4,Station E,40.760,-73.995


In [22]:
# start_df = trip_demo.merge(station_coordinates, how='left', left_on='start_station', right_on='station')
# start_df.rename(columns={
#         "lat": "start_lat",
#         "lng": "start_lng"
#     }, inplace=True)

In [11]:
# start_df.head()

In [27]:
trip_demo = (
    trip_demo
    .merge(
        station_coordinates,
        left_on="start_station",
        right_on="station",
        how="left"
    )
    .rename(columns={
        "lat": "start_lat",
        "lng": "start_lng"
    })
    .drop(columns="station")
    .merge(
        station_coordinates,
        left_on="end_station",
        right_on="station",
        how="left"
    )
    .rename(columns={
        "lat": "end_lat",
        "lng": "end_lng"
    })
    .drop(columns="station")
)



In [28]:
trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,start_lat,start_lng,end_lat,end_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.735,-73.991,40.751,-73.977
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.735,-73.991,40.751,-73.977
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.735,-73.991,40.751,-73.977
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.735,-73.991,40.742,-73.985
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.735,-73.991,40.742,-73.985


In [29]:
map_center = [
    pd.concat([trip_demo["start_lat"], trip_demo["end_lat"]]).mean(),
    pd.concat([trip_demo["start_lng"], trip_demo["end_lng"]]).mean()
]

In [30]:
map_center

[np.float64(40.7427), np.float64(-73.9841)]

In [31]:
trip_demo["duration_min"] = (
    trip_demo["ended_at"] - trip_demo["started_at"]
).dt.total_seconds() / 60

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "started_at",
    "ended_at",
    "duration_min"
]].head()

,ride_id,start_station,end_station,started_at,ended_at,duration_min
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,10.0
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,13.0
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,12.0
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,18.0
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,15.0


In [32]:
trip_demo["date"] = trip_demo["started_at"].dt.date
trip_demo["hour"] = trip_demo["started_at"].dt.hour
trip_demo["day_name"] = trip_demo["started_at"].dt.day_name()
trip_demo["month_name"] = trip_demo["started_at"].dt.month_name()

trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,start_lat,start_lng,end_lat,end_lng,duration_min,date,hour,day_name,month_name
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.735,-73.991,40.751,-73.977,10.0,2025-01-01,8,Wednesday,January
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.735,-73.991,40.751,-73.977,13.0,2025-01-01,8,Wednesday,January
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.735,-73.991,40.751,-73.977,12.0,2025-01-01,8,Wednesday,January
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.735,-73.991,40.742,-73.985,18.0,2025-01-01,9,Wednesday,January
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.735,-73.991,40.742,-73.985,15.0,2025-01-01,9,Wednesday,January


## Folium

### Point Data

In [17]:
m = folium.Map(
    location=[40.74, -73.99],
    zoom_start=13
)

for _, row in station_coordinates.iterrows():
    folium.Marker(
        location=[row["lat"], row["lng"]],
        popup=row["station"]
    ).add_to(m)

m

### Line Data

In [44]:
m = folium.Map(
    location=[40.74, -73.99],
    zoom_start=13
)

start_point = [trip_demo.loc[0, "start_lat"], trip_demo.loc[0, "start_lng"]]
end_point = [trip_demo.loc[0, "end_lat"], trip_demo.loc[0, "end_lng"]]

folium.Marker(start_point, popup="Start").add_to(m)
folium.Marker(end_point, popup="End").add_to(m)

folium.PolyLine(
    locations=[start_point, end_point],
    weight=5,
    opacity=0.8
).add_to(m)

m

### Flow Data

In [34]:
flow_data = (
    trip_demo
    .groupby(
        ["start_station", "end_station"],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count"),
        avg_duration_min=("duration_min", "mean"),
        start_lat=("start_lat", "mean"),
        start_lng=("start_lng", "mean"),
        end_lat=("end_lat", "mean"),
        end_lng=("end_lng", "mean")
    )
    .sort_values("number_of_rides", ascending=False)
)

flow_data

,start_station,end_station,number_of_rides,avg_duration_min,start_lat,start_lng,end_lat,end_lng
0,Station A,Station B,3,11.666667,40.735,-73.991,40.751,-73.977
1,Station A,Station C,2,16.500000,40.735,-73.991,40.742,-73.985
3,Station B,Station A,2,13.500000,40.751,-73.977,40.735,-73.991
2,Station A,Station E,1,28.000000,40.735,-73.991,40.760,-73.995
4,Station B,Station C,1,15.000000,40.751,-73.977,40.742,-73.985
5,Station B,Station D,1,18.000000,40.751,-73.977,40.728,-73.970
6,Station B,Station E,1,25.000000,40.751,-73.977,40.760,-73.995
7,Station C,Station A,1,18.000000,40.742,-73.985,40.735,-73.991
8,Station C,Station B,1,14.000000,40.742,-73.985,40.751,-73.977
9,Station C,Station D,1,20.000000,40.742,-73.985,40.728,-73.970


In [2]:
import geopandas as gpd

In [35]:
start_points = gpd.GeoDataFrame(
    trip_demo.copy(),
    geometry=gpd.points_from_xy(
        trip_demo["start_lng"],
        trip_demo["start_lat"]
    ),
    crs="EPSG:4326"
)

end_points = gpd.GeoDataFrame(
    trip_demo.copy(),
    geometry=gpd.points_from_xy(
        trip_demo["end_lng"],
        trip_demo["end_lat"]
    ),
    crs="EPSG:4326"
)

In [36]:
start_points[[
    "ride_id",
    "start_station",
    "end_station",
    "geometry"
]].head()

,ride_id,start_station,end_station,geometry
0,1,Station A,Station B,POINT (-73.991 40.735)
1,2,Station A,Station B,POINT (-73.991 40.735)
2,3,Station A,Station B,POINT (-73.991 40.735)
3,4,Station A,Station C,POINT (-73.991 40.735)
4,5,Station A,Station C,POINT (-73.991 40.735)


In [37]:
projected_crs = "EPSG:32618"

start_points_projected = start_points.to_crs(projected_crs)
end_points_projected = end_points.to_crs(projected_crs)

In [38]:
trip_demo["distance_m"] = start_points_projected.geometry.distance(
    end_points_projected.geometry
)

trip_demo["distance_km"] = trip_demo["distance_m"] / 1000

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "duration_min",
    "distance_m",
    "distance_km"
]].head()

,ride_id,start_station,end_station,duration_min,distance_m,distance_km
0,1,Station A,Station B,10.0,2133.621698,2.133622
1,2,Station A,Station B,13.0,2133.621698,2.133622
2,3,Station A,Station B,12.0,2133.621698,2.133622
3,4,Station A,Station C,18.0,927.671157,0.927671
4,5,Station A,Station C,15.0,927.671157,0.927671


In [39]:
trip_demo[["distance_m", "distance_km"]].describe()

,distance_m,distance_km
count,20.000000,20.000000
mean,2113.717746,2.113718
std,898.892579,0.898893
min,927.671157,0.927671
25%,1665.451089,1.665451
50%,2133.621698,2.133622
75%,2282.181051,2.282181
max,4132.274455,4.132274


## Citi Bike Jersey

In [5]:
# eval: true
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile


### Downloading the Citi Bike Data for 202510

In [ ]:
CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
PERIOD = "202510"

In [4]:
file_name = f"JC-{PERIOD}-citibike-tripdata.zip"
url = f"{CITIBIKE_INDEX_URL}/{file_name}"
url

'https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip'

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)
zip_path = output_dir / file_name

In [ ]:
## Dowloading the Zip file
url = f"{CITIBIKE_INDEX_URL}/{file_name}"

print(f"Downloading: {url}")
urlretrieve(url, zip_path)

print(f"Saved ZIP file to: {zip_path}")


with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)

print(f"Extracted files into: {output_dir}")

Downloading: https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip
Saved ZIP file to: ../data/citibike/JC-202510-citibike-tripdata.zip
Extracted files into: ../data/citibike


In [8]:
zip_path.unlink()
print("ZIP file removed.")

ZIP file removed.


In [41]:
file_name  = 'JC-202510-citibike-tripdata.csv'
citibike_202510 = pd.read_csv(f'{output_dir}/{file_name}')
citibike_202510.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,929852F0DC04F49C,electric_bike,2025-10-08 06:59:53.767,2025-10-08 07:04:08.012,Warren St,JC006,Essex Light Rail,NaN,40.721124,-74.038051,NaN,NaN,member
1,7E6F8FADF45B1D8E,electric_bike,2025-10-08 08:13:15.096,2025-10-08 08:18:16.168,Newark Ave,JC032,Essex Light Rail,NaN,40.721525,-74.046305,NaN,NaN,member
2,9B679869FCA8F845,electric_bike,2025-10-09 10:10:26.020,2025-10-09 10:20:08.505,Liberty Light Rail,JC052,Newport PATH,NaN,40.711242,-74.055701,NaN,NaN,member
3,E2051DD457BC4076,electric_bike,2025-10-09 11:34:22.937,2025-10-09 11:48:49.501,Hoboken Ave at Monmouth St,JC105,Newport PATH,NaN,40.735208,-74.046964,NaN,NaN,member
4,239AC07371E414EC,electric_bike,2025-10-23 19:24:16.237,2025-10-23 19:28:22.592,Union St & Bergen Ave,JC122,Garfield Light Rail,JC119,40.715750,-74.078870,40.710526,-74.070112,member


In [9]:
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    year list of strings
    """
    YEAR = year
    MONTH =  [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]

    periods = []

    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
    # print(periods)
    return periods

PERIODS = period_iterator(["2025"],0,12)
PERIODS

['202501',
 '202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [10]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import requests

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

for i in PERIODS:
    urls = [
    f"{CITIBIKE_INDEX_URL}/JC-{i}-citibike-tripdata.csv.zip",
    f"{CITIBIKE_INDEX_URL}/JC-{i}-citibike-tripdata.zip",
]

    last_error = None

    for url in urls:
        try:
            print(f"Trying: {url}")

            response = requests.get(url, timeout=30)
            response.raise_for_status()

            with ZipFile(BytesIO(response.content), "r") as zip_ref:
                zip_ref.extractall(output_dir)

            print(f"Success: {url}")
            break

        except Exception as e:
            print(f"Failed: {url}")
            last_error = e

    else:
        raise RuntimeError(f"Both URLs failed for period {i}") from last_error

Trying: https://s3.amazonaws.com/tripdata/JC-202501-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202501-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202502-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202502-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202503-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202503-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202504-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202504-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202505-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202505-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202506-citibike-tripdata.csv.zip
Success: https://s3.amazonaws.com/tripdata/JC-202506-citibike-tripdata.csv.zip
Trying: https://s3.amazonaws.com/tripdata/JC-202507-citibi

### Removing __MACOSX

In [42]:
import shutil

shutil.rmtree(output_dir / "__MACOSX")